# Rule-based classification of the preprocessed buildings

**Input (read only):** `data/output/04_buildings_enriched.gpkg` (layers `buildings`, `building_pois`, written by pipeline step 04) and `data/output/01_all_pois.gpkg` (layer `pois`, step 01) — paths in `rules/config.py`
**Output:** `data/output/rules_classification.parquet` — one row per building: its Bosserhof class, its MiD activity labels and the rule layer that decided

Runs `rule_utils.classify_building()` — the rule tables unchanged, with their own weights — on every building the preprocessing kept (39,786 in the Braunschweig region). Every answer is keyed on the preprocessing's `building_id`, so it joins one-to-one to its buildings.

**How the preprocessed data become the engine's input:**

1. **POI tags.** The POIs come from the POI placement (04 `building_pois`), their tags from the POI table (01 `pois`). Each POI gives every key the engine reads: `amenity`, `shop`, `tourism`, `building`, `information`, and `office` / `craft` / `social_facility` as `additional_information`, each from its column in 01, else from the JSON of the POI's other tags (`information` lives only there). The preprocessing's POI filters stand; no further filter is applied.
2. **Two runs.** `sites` (main): the POIs placed on the building plus the site polygons it lies in (campus, plant, estate). `own`: the POIs placed on the building only, as a sensitivity check.
3. **Order.** Where the engine has to pick one value, the building's main occupant wins: own POIs before sites, a larger share before a smaller one, and the POI id breaks ties. The engine keeps the *first* of equally weighted classes but the *last* value of a repeated `office` / `craft` / `social_facility` key, so `additional_information` is de-duplicated and then reversed.
4. **Building fields.** ALKIS code: 04 `function` (none on the OSM gap-fill rows, whose placeholder `OSM` is no code). Footprint type: the OSM twin's tag on ALKIS rows, the footprint's own tag on OSM rows. Land use: `osm_landuse`. Three codes have no ALKIS rule (31001_2720 agricultural, 31001_2410 road-traffic operations, 31001_3073 barracks), because the rules were written for data from which such buildings had been removed; here they are decided by their POI tags, else by the footprint tag or the land use.
5. **Nothing else.** No names, no sizes, no change to a rule or a weight.


In [1]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('.').resolve()))   # rules/: its own config and rule_utils
from config import *

import hashlib
import json
import re
import time

import pandas as pd
import pyogrio

from rule_utils import classify_building, reachable_bosserhof_classes


def _sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()[:12]


print('Config loaded')
for _f in (ENRICHED_BUILDINGS_FILE, ALL_POIS_FILE, ROOT / 'rule_utils.py'):
    print(f'  {_sha(_f)}  {_f.name}')

Config loaded
  4fd662121070  04_buildings_enriched.gpkg
  1e59d2bd360f  01_all_pois.gpkg
  80b9f326fc91  rule_utils.py


## 1. Load the buildings, the POI placement and the POI tags

In [2]:
B_COLS = ['building_id', 'source', 'function', 'osm_twin_tag', 'osm_building', 'osm_landuse']
P_COLS = ['building_id', 'poi_id', 'poi_role', 'share_in_building', 'share_of_site']
TAG_COLS = ['amenity', 'shop', 'tourism', 'building']          # one value per POI, looked up in the tag tables
KV_KEYS = ['office', 'craft', 'social_facility']               # read through additional_information

# The input is only read: its GeoPackages are opened immutable, so SQLite writes no side files next to them
buildings = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='buildings', read_geometry=False, columns=B_COLS,
                                   IMMUTABLE='YES')
pairs = pyogrio.read_dataframe(ENRICHED_BUILDINGS_FILE, layer='building_pois', read_geometry=False, columns=P_COLS,
                               IMMUTABLE='YES')
pois = pyogrio.read_dataframe(ALL_POIS_FILE, layer='pois', read_geometry=False,
                              columns=['poi_id', *TAG_COLS, *KV_KEYS, 'tags'], IMMUTABLE='YES')

assert buildings['building_id'].is_unique, 'building_id is not unique in 04'
assert pairs['building_id'].isin(buildings['building_id']).all(), 'a POI pair points at a building 04 does not hold'
assert not pairs.duplicated(['building_id', 'poi_id']).any(), 'a building-POI pair occurs twice'
assert pois['poi_id'].is_unique, 'poi_id is not unique in 01'
assert pairs['poi_id'].isin(pois['poi_id']).all(), 'a POI pair points at a POI 01 does not hold'
_roles = set(pairs['poi_role'])
assert _roles <= set(POI_RUNS['sites']), f'unexpected poi_role values: {_roles - set(POI_RUNS["sites"])}'

print(f'{len(buildings):,} buildings: {buildings["source"].value_counts().to_dict()}')
print(f'{len(pairs):,} building-POI pairs: {pairs["poi_role"].value_counts().to_dict()}')
print(f'{len(pois):,} POIs in 01')

39,786 buildings: {'alkis': 38186, 'osm': 1600}
41,285 building-POI pairs: {'site': 18652, 'point': 10861, 'footprint': 10146, 'unit': 1626}
27,843 POIs in 01


## 2. Per POI: the values the engine reads

In [3]:
def _text(v):
    """a tag value as plain str, or None - never NaN or pd.NA (the engine's land-use test raises on pd.NA)"""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    v = str(v).strip()
    return v or None


def _tags(tags):
    """the JSON of a POI's remaining tags: information=* lives only there, and pyrosm left a few other keys there"""
    return json.loads(tags) if isinstance(tags, str) and tags else {}


# a key's value from its column, else from the JSON (plain lists: a Series would turn None into NaN)
_tj = [_tags(t) for t in pois['tags']]
poi_vals = pd.DataFrame({'poi_id': pois['poi_id']})
for _c in TAG_COLS:
    poi_vals[_c] = [_text(v) or _text(j.get(_c)) for v, j in zip(pois[_c], _tj)]
poi_vals['information'] = [_text(j.get('information')) for j in _tj]
_kv = {k: [_text(v) or _text(j.get(k)) for v, j in zip(pois[k], _tj)] for k in KV_KEYS}
_json_only = {c: sum(not isinstance(v, str) and isinstance(_text(j.get(c)), str) for v, j in zip(pois[c], _tj))
              for c in [*TAG_COLS, *KV_KEYS]}
print(f'values found only in the JSON: {_json_only}')
poi_vals['additional_information'] = [
    '; '.join(f'{k}: {v}' for k, v in zip(KV_KEYS, vals) if v is not None) or None
    for vals in zip(*_kv.values())
]
LIST_COLS = [*TAG_COLS, 'information']

print('POIs in 01 carrying each value the engine reads:')
for _c in [*LIST_COLS, *KV_KEYS]:
    _n = poi_vals[_c].notna().sum() if _c in poi_vals else sum(v is not None for v in _kv[_c])
    print(f'  {_c:<16}{int(_n):>7,}')

values found only in the JSON: {'amenity': 0, 'shop': 0, 'tourism': 2, 'building': 0, 'office': 0, 'craft': 0, 'social_facility': 0}
POIs in 01 carrying each value the engine reads:
  amenity           9,595
  shop              6,295
  tourism           2,013
  building          9,023
  information          71
  office            1,704
  craft               684
  social_facility     373


## 3. Per building: the engine's input row, for each run

In [4]:
# Plain Python from here on: pandas 3 stores missing text as NaN, and the engine is fed lists and str, as it expects
BASE = [{'gml_id': b['building_id'],
         'function': None if b['source'] == 'osm' else _text(b['function']),
         'osm_building_type': _text(b['osm_twin_tag']) if b['source'] == 'alkis' else _text(b['osm_building']),
         'osm_landuse_class': _text(b['osm_landuse'])}
        for b in buildings.to_dict('records')]
POI_COLS = [*LIST_COLS, 'additional_information']


def _unique(values):
    """the values in order, each once; None when there is none"""
    out = list(dict.fromkeys(values))
    return out or None


def engine_rows(roles):
    """One input row per building, in the order of BASE: the POI columns as plain lists, main occupant first"""
    p = pairs[pairs['poi_role'].isin(roles)].merge(poi_vals, on='poi_id', how='left')
    p['_site'] = (p['poi_role'] == 'site').astype(int)
    p['_share'] = p['share_in_building'].where(p['_site'] == 0, p['share_of_site']).fillna(0.0)
    p = p.sort_values(['_site', '_share', 'poi_id'], ascending=[True, False, True], kind='stable')
    per = {}
    for bid, *vals in p[['building_id', *POI_COLS]].itertuples(index=False, name=None):
        acc = per.setdefault(bid, [[] for _ in POI_COLS])
        for a, v in zip(acc, vals):
            if isinstance(v, str):
                a.append(v)
    rows = []
    for b in BASE:
        acc = per.get(b['gml_id'], [[] for _ in POI_COLS])
        r = dict(b)
        for c, vals in zip(POI_COLS, acc):
            u = _unique(vals)
            # the engine keeps the LAST value of a repeated office / craft / social_facility key: de-duplicated first,
            # then reversed, so the main occupant's value comes last and wins
            r[c] = u[::-1] if u and c == 'additional_information' else u
        rows.append(r)
    return rows


_NAN = re.compile(r'(^|:\s*)(nan|nat|none|null|<na>)\s*(;|$)', re.IGNORECASE)    # a missing value written out as text


def check_rows(rows):
    """the engine reads lists and str only: a numpy array silently misses (ast joins its items), pd.NA raises,
    and a missing value written out as text ('nan', 'office: nan') would pass for a tag"""
    for r in rows:
        for c in POI_COLS:
            v = r[c]
            assert v is None or (type(v) is list and v and all(type(x) is str for x in v)), (r['gml_id'], c, v)
            assert v is None or not any(_NAN.search(x) for x in v), (r['gml_id'], c, v)
        for c in ('function', 'osm_building_type', 'osm_landuse_class'):
            assert r[c] is None or (type(r[c]) is str and not _NAN.search(r[c])), (r['gml_id'], c, r[c])


ROWS = {}
for run, roles in POI_RUNS.items():
    ROWS[run] = engine_rows(roles)
    check_rows(ROWS[run])
    _n = sum(any(r[c] is not None for c in POI_COLS) for r in ROWS[run])
    print(f'[{run}] {len(ROWS[run]):,} rows; {_n:,} buildings carry at least one POI value the engine reads')

_ex = next(r for r in ROWS['sites'] if r['shop'] and r['amenity'])
print()
print('one input row (sites run):')
for _k, _v in _ex.items():
    print(f'  {_k:<24}{_v}')

[sites] 39,786 rows; 18,596 buildings carry at least one POI value the engine reads


[own] 39,786 rows; 16,617 buildings carry at least one POI value the engine reads

one input row (sites run):
  gml_id                  DENIAL01000002vN
  function                31001_1000
  osm_building_type       yes
  osm_landuse_class       residential
  amenity                 ['restaurant']
  shop                    ['bakery']
  tourism                 None
  building                None
  information             None
  additional_information  None


## 4. Classify

In [5]:
REACHABLE = reachable_bosserhof_classes()
results = {}
for run, rows in ROWS.items():
    _t = time.time()
    res = pd.DataFrame([classify_building(r) for r in rows])
    assert res['gml_id'].tolist() == [b['gml_id'] for b in BASE], 'the answers lost the building order'
    assert res['bosserhof_class'].dropna().str.lower().isin(REACHABLE).all(), 'a class no rule can emit'
    assert res['mid_labels'].map(lambda ls: set(ls) <= TARGET_MID_LABELS).all(), 'a label outside the 12'
    assert (res['bosserhof_class'].notna() == res['mid_labels'].map(lambda ls: 'work' in ls)).all(), 'work without a class'
    results[run] = res
    print(f'[{run}] {len(res):,} buildings classified in {time.time() - _t:.1f} s')

_layers = pd.DataFrame({run: res['interpreted_type'].value_counts() for run, res in results.items()}).fillna(0).astype(int)
_layers.loc['total'] = _layers.sum()
print()
print('which layer decided (buildings):')
print(_layers.to_string())
print()
for run, res in results.items():
    _no = res[res['bosserhof_class'].isna()]
    print(f'[{run}] a class on {int(res["bosserhof_class"].notna().sum()):,}; none on {len(_no):,}: '
          f'{_no["interpreted_type"].value_counts().to_dict()} (alkis_code: an ALKIS code the rules mark as no activity)')

[sites] 39,786 buildings classified in 0.2 s


[own] 39,786 buildings classified in 0.3 s

which layer decided (buildings):
                  sites    own
interpreted_type              
alkis_code        21683  23464
poi_tags          16994  15059
osm_fallback        876    941
no_signal           233    322
total             39786  39786

[sites] a class on 37,732; none on 2,054: {'alkis_code': 1821, 'no_signal': 233} (alkis_code: an ALKIS code the rules mark as no activity)
[own] a class on 37,439; none on 2,347: {'alkis_code': 2025, 'no_signal': 322} (alkis_code: an ALKIS code the rules mark as no activity)


In [6]:
_cls = pd.DataFrame({run: res['bosserhof_class'].value_counts() for run, res in results.items()}).fillna(0).astype(int)
_cls = _cls.sort_values('sites', ascending=False)
print(f'Bosserhof classes ({len(_cls)} occur):')
print(_cls.to_string())
print()
_lab = pd.DataFrame({run: pd.Series([l for ls in res['mid_labels'] for l in ls]).value_counts()
                     for run, res in results.items()}).fillna(0).astype(int)
print('MiD labels (buildings carrying each):')
print(_lab.sort_values('sites', ascending=False).to_string())
print()
_s, _o = results['sites'], results['own']
_dc = _s['bosserhof_class'].fillna('') != _o['bosserhof_class'].fillna('')
_dl = _s['mid_labels'].map(tuple) != _o['mid_labels'].map(tuple)
print(f'sites vs own: the class differs on {int(_dc.sum()):,} buildings, the labels on {int(_dl.sum()):,}')
_moves = (_o.loc[_dc, 'bosserhof_class'].fillna('(none)') + '  ->  ' + _s.loc[_dc, 'bosserhof_class'].fillna('(none)'))
print('the most frequent changes (own -> sites):')
print(_moves.value_counts().head(12).to_string())

Bosserhof classes (39 occur):
                                                                  sites   own
bosserhof_class                                                              
services                                                           9334  9672
industrial operations production                                   9008  9117
public facilities                                                  3736  3917
retail small scale                                                 2460  2444
customer oriented services                                         2282  2262
restaurants gastronomy                                             1820  1820
schools                                                            1373  1155
hotels                                                              901   854
kindergartens                                                       709   623
normal office                                                       574   559
facilities for culture leisure and

## 5. Save

In [7]:
out = pd.DataFrame({'building_id': [b['gml_id'] for b in BASE]})
for run, res in results.items():
    out[f'cls_{run}'] = res['bosserhof_class'].to_numpy()
    out[f'labels_{run}'] = res['mid_labels'].map(';'.join).to_numpy()      # sorted by the engine; '' = no activity
    out[f'layer_{run}'] = res['interpreted_type'].to_numpy()
out.to_parquet(RULES_OUTPUT_FILE, index=False)
print(f'Saved {len(out):,} rows x {out.shape[1]} columns -> {RULES_OUTPUT_FILE.relative_to(REPO_ROOT)}')
print(out.head().to_string())

Saved 39,786 rows x 7 columns -> data\output\rules_classification.parquet
        building_id          cls_sites                   labels_sites layer_sites            cls_own                     labels_own   layer_own
0  DENIAL01000002A1           services                  business;work  alkis_code           services                  business;work  alkis_code
1  DENIAL01000002Ag  public facilities                           work    poi_tags  public facilities                           work    poi_tags
2  DENIAL01000002Bm      kindergartens                 childcare;work    poi_tags      kindergartens                 childcare;work    poi_tags
3  DENIAL01000002CF      kindergartens  childcare;leisure;meetup;work    poi_tags      kindergartens  childcare;leisure;meetup;work    poi_tags
4  DENIAL01000002CX               None                                 alkis_code               None                                 alkis_code
